<div style="background-color: black; color: white; padding: 10px;text-align: center;">
  <strong>Date Published:</strong> August 16, 2026 <strong>Author:</strong> Adnan Alaref
</div>

#  **Baseline B5**  
  * Temporal on crops (LSTM on player level)
  * Similar 2 implementation paths to build representation per person
  * You can represent each clip with the last hidden state
  * Max pool all players representations (9 per image)
  * Then do the NN network exactly like in B3 (on images)  
  >  Features classifier - no temporal info

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 1: Import Library.</div>

In [16]:
# !uv pip uninstall --system -y torch torchvision torchaudio >/dev/null 2>&1

In [17]:
# !uv pip install --system torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0  >/dev/null 2>&1

In [18]:
# !uv pip install --system numpy==1.26.4 pandas==2.2.2 seaborn==0.13.2 --no-cache-dir >/dev/null 2>&1

In [19]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

2.2.0+cu121
12.1
Tesla P100-PCIE-16GB
(6, 0)


In [20]:
import numpy as np
import pandas as pd
import seaborn as sns

print(np.__version__)
print(pd.__version__)
print(sns.__version__)

1.26.4
2.2.2
0.13.2


In [21]:
import re
import os
import cv2
import torch
import random
import torch.nn as nn
from pathlib import Path
from pprint import pprint
from tqdm.auto import tqdm
from PIL import Image, ImageOps
from collections import Counter
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import f1_score, classification_report
from typing import Dict, List, TypeVar, Tuple, Any, Callable, Optional

import warnings
warnings.simplefilter(action='ignore')
warnings.filterwarnings(action='ignore', category=FutureWarning)

In [22]:
dataset_root = r"/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball"

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 2: Collect Player Temporal Data From Clips With 9 Frames Include Middle Image.</div>

In [23]:
from collections import OrderedDict
from dataclasses import dataclass

# ── Data structure for a single player's temporal sequence ────────────────────
@dataclass
class PlayerInfo:
  player_id: int
  player_action: str
  frame_paths: List[Path]
  player_boxes : List[Tuple[int, int, int, int]]

# ── Collect player's temporal sequences from the dataset ──────────────────────
def collect_player_temporal_sequences(dataset_root: str,
                                      split_group:List[int])-> OrderedDict[Tuple[int, int ,int], PlayerInfo]:
  """
    Collect 9-frame temporal sequences for each player in the given videos.

    For each annotated clip, the function collects the player's bounding boxes
    from four frames before the target frame through four frames after it.
    The player's action is taken from the target frame.

    Args:
        dataset_root: Root directory of the volleyball dataset.
        split_group: List of video IDs to process.

    Returns:
        An OrderedDict mapping (video_id, clip_id, player_id) tuples
        to PlayerInfo objects.
  """

  player_sequences: OrderedDict[Tuple[int, int ,int], PlayerInfo] = OrderedDict()

  videos_root = os.path.join(dataset_root, "videos")
  videos_tracking_root = os.path.join(dataset_root,"volleyball_tracking_annotation")

  # ── Iterate over videos in the requested split ──────────────────────────────
  for video_id in split_group:

    # Path to annotations.txt for the current video
    video_annot = os.path.join(videos_root, str(video_id), 'annotations.txt')
    if not os.path.isfile(video_annot):
      raise FileNotFoundError(f"Video annotation file does not exist.: \
                              \nPath: {video_annot}")

    # ── Read each annotated clip ──────────────────────────────────────────────
    with open(file= video_annot, mode='r') as file_annot:
      for annot_line in file_annot:
        clip_id = int(annot_line.partition(".")[0])

        path = Path(videos_root, str(video_id), str(clip_id))
        video_track_annotation = os.path.join(videos_tracking_root, str(video_id)
                                                 ,str(clip_id), f"{clip_id}.txt")

        begin_frame = clip_id - 4
        end_frame = clip_id + 4
        selected_frames = [frame for frame in range(begin_frame, end_frame + 1)]

        current_player_id = None
        player_action = None
        player_boxes: List[Tuple[int, int, int, int]] = []

        # ── Read tracking annotations for the current clip ────────────────────
        with open(file= video_track_annotation, mode='r') as file_tracker:
          # 0 31 386 85 511 13276 0 0 0 standing
          for tracker_line in file_tracker:

            parts = tracker_line.strip().split(" ")
            player_id = int(parts[0])
            frame_id = int(parts[5])

            # ── Detect a new player ───────────────────────────────────────────
            if current_player_id is None:
              current_player_id = player_id


            elif player_id != current_player_id:

              # ── Save previous player ────────────────────────────────────────
              player_sequences[(video_id, clip_id, current_player_id)] = PlayerInfo(
                player_id= current_player_id,
                player_action= player_action,
                frame_paths=[
                  path / f"{frame}.jpg"
                  for frame in selected_frames
                ],
                player_boxes= player_boxes
              )

              # ── Start new player ────────────────────────────────────────────
              current_player_id = player_id
              player_action = None
              player_boxes = []

            # ── Collect only the 9 temporal frames ────────────────────────────
            if begin_frame <= frame_id <= end_frame:
              bbox = tuple(map(int, parts[1:5]))
              player_boxes.append(bbox)

              if frame_id == clip_id:
                player_action  = parts[-1] # parts[9]

          # ── Save the last player ────────────────────────────────────────────
          player_sequences[(video_id, clip_id, current_player_id)] = PlayerInfo(
            player_id= current_player_id,
            player_action= player_action,
            frame_paths=[
              path / f"{frame}.jpg"
              for frame in selected_frames
            ],
            player_boxes= player_boxes
          )

  return player_sequences

In [24]:
df = collect_player_temporal_sequences(dataset_root, [0])
pprint(df[(0,13286,0)])

PlayerInfo(player_id=0,
           player_action='standing',
           frame_paths=[PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/13286/13282.jpg'),
                        PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/13286/13283.jpg'),
                        PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/13286/13284.jpg'),
                        PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/13286/13285.jpg'),
                        PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/13286/13286.jpg'),
                        PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/13286/13287.jpg'),
                        PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/13286/13288.jpg'),
        

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 3: Create Custom Dateset.</div>

In [25]:
def build_samples(dataset: OrderedDict)-> List[
                                            Tuple[
                                              Tuple[int, int, int],
                                              str,
                                              List[Path],
                                              List[Tuple[int, int, int, int]]
                                            ]
                                          ]:
  """
    Build a list of player temporal samples from the dataset.

    Args:
        dataset: OrderedDict containing PlayerInfo objects indexed by
            (video_id, clip_id, player_id).

    Returns:
        A list containing the sample key, player action, frame paths,
        and player bounding boxes for each player temporal sequence.
  """

  return [
          ( key,
            value.player_action,
            value.frame_paths,
            value.player_boxes
          )
          for key, value in dataset.items()
  ]

In [26]:
class PlayerTemporalSequenceDataset(Dataset):
  # Fixed classes list
  PLAYER_ACTIVITY_CLASSES = ["standing", "moving", "waiting", "blocking",
                               "digging", "setting", "falling", "spiking", "jumping"
  ]

  def __init__(self, player_sequences: OrderedDict[Tuple[int, int, int], PlayerInfo], transform = None, debug:bool = False)->None:
    super().__init__()
    self.debug = debug
    self.transform = transform
    self.samples = build_samples(player_sequences)
    self.num_crops = sum(len(bboxes) for _, _, _, bboxes in self.samples)

    # Class mappings
    self.player_action2idx = {action:idx for idx, action in enumerate(self.PLAYER_ACTIVITY_CLASSES)}
    self.player_idx2action = {idx:action for action , idx in self.player_action2idx.items()}

    # Encode labels once
    self.labels = [self.player_action2idx[p_action] for _, p_action, _, _ in self.samples]

  def __len__(self)->int:
    """Return the number of player temporal sequence."""
    return len(self.samples)

  def __getitem__(self, index)->Tuple[torch.Tensor, torch.Tensor]:

    _, p_action, frame_paths, player_boxes = self.samples[index]

    player_crops:List[Image.Image] = []
    assert len(frame_paths) == len(player_boxes) == 9,(
      f"Expected 9 frames and boxes, "
      f"got {len(frame_paths)} frames and {len(player_boxes)} boxes."
    )

    for frame_path, box in zip(frame_paths, player_boxes):
      try:
        with Image.open(frame_path) as image:
          image = ImageOps.exif_transpose(image) # safe, no-op here
          image = image.convert("RGB")

        # ── Crop player from the current frame ────────────────────────────────
        player_crop = image.crop(box)

        # ── Collect processed player crop ─────────────────────────────────────
        player_crops.append(player_crop)
          
      except Exception as e:
        raise RuntimeError(f"Failed to load image: {frame_path}") from e

    # ── Apply temporal transform to all 9 crops ───────────────────────────────
    if self.transform is not None:
      crops = self.transform(player_crops)

    else:
      crops = torch.stack([
          TF.to_tensor(crop)
          for crop in player_crops
      ])

    # ── Encode player action label ────────────────────────────────────────────
    label = torch.tensor(self.labels[index], dtype= torch.long)

    if self.debug:
      print(f"\nSample : [{index}]")
      print("transform:", type(self.transform))
      print("player_crops:", type(player_crops))
      print("number:", len(player_crops))
      print("crop types:", [type(x) for x in player_crops])
        
      print(f"Player Action : {p_action}")
      print(f"Crops Shape   : {crops.shape}")
      print(f"Label         : {label}")
      print("\nPlayer Boxes  :")
      pprint(player_boxes)
      print("\nFrame Paths   :")
      pprint(frame_paths)
      print("\nTensors Data Crops and Label:")

    return crops, label

  def summary(self)-> None:
    action_map = Counter(action for _, action, _, _ in self.samples)
    print("="*40)
    print(f"Total Crops       : {self.num_crops}")
    print(f"Total Samples     : {len(self.samples)}")
    print(f"Unique actions    : {len(self.PLAYER_ACTIVITY_CLASSES)}")
    print("-"*40)
    print("Player actions frequency:")
    for p_action, count in sorted(action_map.items(), key=lambda item:item[1], reverse=True):
      print(f"  {p_action:<10} : {count}")
    print("="*40)

## **Step 3.1: Debug Temporal Transform with PIL Crop List**

In [27]:
from torchvision import transforms

# Create a custom transform class that processes a list of PIL Images
class TestComposeForList:
    """
    Debug transform that applies standard torchvision
    transforms independently to each image in a temporal sequence.
    """
    def __init__(self, transforms_list):
        self.transforms = transforms_list

    def __call__(self, img_list: List[Image.Image]) -> torch.Tensor:
        transformed_crops = []
        for img in img_list:
            # Apply each transform in the list to the individual image
            for t in self.transforms:
                img = t(img)
            transformed_crops.append(img)
        # Stack all transformed images into a single tensor
        return torch.stack(transformed_crops)

# ── Test Custom Dataset ───────────────────────────────────────────────────
# Use the custom TestComposeForList to apply transformations to each image in the list
pp = PlayerTemporalSequenceDataset(df, transform=TestComposeForList([
    transforms.Resize((128, 128)), # Resize to a consistent (height, width)
    transforms.ToTensor()          # Convert PIL Image to PyTorch Tensor
]), debug=True)
pp[0]


Sample : [0]
transform: <class '__main__.TestComposeForList'>
player_crops: <class 'list'>
number: 9
crop types: [<class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>, <class 'PIL.Image.Image'>]
Player Action : falling
Crops Shape   : torch.Size([9, 3, 128, 128])
Label         : 6

Player Boxes  :
[(366, 440, 462, 575),
 (367, 440, 462, 575),
 (367, 440, 461, 575),
 (367, 440, 460, 575),
 (367, 440, 459, 575),
 (368, 440, 458, 575),
 (368, 440, 454, 575),
 (368, 440, 451, 575),
 (369, 440, 448, 575)]

Frame Paths   :
[PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/48075/48071.jpg'),
 PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/48075/48072.jpg'),
 PosixPath('/kaggle/input/datasets/sherif31/group-activity-recognition-volleyball/videos/0/480

(tensor([[[[0.4510, 0.3529, 0.2902,  ..., 0.9255, 0.9255, 0.9255],
           [0.3137, 0.2784, 0.2588,  ..., 0.9255, 0.9255, 0.9255],
           [0.2353, 0.2431, 0.2588,  ..., 0.9255, 0.9255, 0.9255],
           ...,
           [0.9373, 0.9333, 0.9333,  ..., 0.9137, 0.9137, 0.9176],
           [0.9412, 0.9412, 0.9412,  ..., 0.9176, 0.9176, 0.9176],
           [0.9451, 0.9451, 0.9490,  ..., 0.9294, 0.9294, 0.9294]],
 
          [[0.2039, 0.1059, 0.0392,  ..., 0.6667, 0.6667, 0.6667],
           [0.0706, 0.0353, 0.0118,  ..., 0.6667, 0.6667, 0.6667],
           [0.0000, 0.0039, 0.0118,  ..., 0.6667, 0.6667, 0.6667],
           ...,
           [0.6431, 0.6353, 0.6314,  ..., 0.6235, 0.6235, 0.6275],
           [0.6471, 0.6431, 0.6392,  ..., 0.6275, 0.6275, 0.6275],
           [0.6510, 0.6471, 0.6431,  ..., 0.6392, 0.6392, 0.6392]],
 
          [[0.1412, 0.0510, 0.0000,  ..., 0.5333, 0.5333, 0.5333],
           [0.0275, 0.0118, 0.0000,  ..., 0.5333, 0.5333, 0.5333],
           [0.0000, 0.00

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 4: Experiment Configuration.</div>

In [28]:
from dataclasses import field

@dataclass
class ModelConfig:
  # name: TYPE = VALUE
  dropout_prop: float = 0.5
  player_classes : int = 9
  device: torch.device | None = None
  # mutable → must use field
  train_split:List[int] = field(default_factory=lambda:[
    1, 3, 6, 7, 10, 13, 15, 16, 18, 22, 23, 31,
    32, 36, 38, 39, 40, 41, 42, 48, 50, 52, 53, 54
  ])

  test_split:List[int] = field(default_factory=lambda:[
    4, 5, 9, 11, 14, 20, 21, 25, 29, 34, 35, 37, 43, 44, 45, 47
  ])

  val_split:List[int] = field(default_factory=lambda:[
    0, 2, 8, 12, 17, 19, 24, 26, 27, 28, 30, 33, 46, 49, 51
  ])

  classes_names:List[str] = field(default_factory=lambda:[
    "standing", "moving", "waiting", "blocking",  "digging", "setting", "falling", "spiking", "jumping"
  ])

  def __post_init__(self):
    max_index:int = 54
    if self.device is None:
        self.device = torch.device(
          "mps"
          if torch.backends.mps.is_available()
          else "cuda"
          if torch.cuda.is_available()
          else "cpu"
        )

    # validate splits
    for name, split in {
      "train": self.train_split,
      "test": self.test_split,
      "val": self.val_split,
      }.items():

      # empty
      if len(split) == 0:
        raise ValueError(f"{name}_split cannot be empty.")

      # uniqueness
      if len(split) != len(set(split)):
        raise ValueError(f"{name}_split contains duplicate indices.")

      # range
      if any(idx < 0 or idx > max_index for idx in split):
        raise ValueError(f"{name}_split indices must be in range [0, {max_index}]")

    # data leakage check
    all_index = set(self.train_split) | set(self.val_split) | set(self.test_split)
    if len(all_index) != (len(self.train_split) + len(self.val_split) + len(self.test_split)):
      raise ValueError("Splits overlap, There are data leakage problem!")

@dataclass
class TrainingConfig:
  # Immutable → not use field
  learning_rate: float = 5e-5
  val_bsize:int = 32
  train_bsize: int = 32
  epochs: int = 10
  checkpoint_path: str = r'/kaggle/working/chekpoints'
  eta_min: float = 1e-6
  momentum:float = 0.9
  print_every:int = 1
  weight_decay: float = 1e-4
  betas: tuple = (0.9, 0.999)

model_config = ModelConfig()
train_config = TrainingConfig()

In [29]:
def save_checkpoints(model: nn.Module, scaler: torch.cuda.amp.GradScaler|None,
                     optimizer: torch.optim.Optimizer, train_loss: float, val_loss: float, macro_f1: float, epoch: int, path: str)->None:
  # Define Chekpoint
  checkpoint = {
    "epoch": epoch,
    "macro_f1":float(macro_f1),
    "val_loss": float(val_loss),
    "train_loss": float(train_loss),
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
  }

  # add scaler ONLY if it exists
  if scaler is not None:
    checkpoint['scaler_state_dict'] = scaler.state_dict()

  # Create dir
  os.makedirs(os.path.dirname(path), exist_ok=True)
  torch.save(checkpoint, path)
  print(f"Saved checkpoint: {path}")


def load_checkpoint(path:str, device: str|None=None)->dict:
  if not os.path.isfile(path):
    raise FileNotFoundError(f"Checkpoint file does not found.\nPath: {path}")

  # Load Data
  checkpoint_data = torch.load(path, map_location=device)
  return checkpoint_data

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 5: Dataset and DataLoader Setup.</div>

In [30]:
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF


class TemporalTransform:
    """
    Apply the same random augmentation parameters
    to all 9 frames of a temporal sequence.
    """

    def __init__(
        self,
        resize_size: int = 256,
        crop_size: int = 224,
        rotation: float = 2.0,
        brightness: float = 0.2,
        contrast: float = 0.2,
        saturation: float = 0.2,
        hue: float = 0.05,
        horizontal_flip_p: float = 0.5,
    ) -> None:

        # ── Deterministic spatial transformations
        self.resize = T.Resize(resize_size)
        self.crop = T.CenterCrop(crop_size)

        # ── Augmentation strengths
        self.rotation = rotation
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation
        self.hue = hue
        self.horizontal_flip_p = horizontal_flip_p

        # ── Convert PIL Image to an Image TVTensor
        self.to_image = T.ToImage()

        # ── Convert uint8 [0, 255] to float32 [0, 1]
        self.to_dtype = T.ToDtype(
            torch.float32,
            scale=True,
        )

        # ── ImageNet normalization parameters
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]

    def __call__(self, crops: List[Image.Image])-> torch.Tensor:

        # ── Sample augmentation parameters ONCE for the entire sequence
        angle = torch.empty(1).uniform_(
            -self.rotation,
            self.rotation,
        ).item()

        brightness_factor = torch.empty(1).uniform_(
            max(0.0, 1.0 - self.brightness),
            1.0 + self.brightness,
        ).item()

        contrast_factor = torch.empty(1).uniform_(
            max(0.0, 1.0 - self.contrast),
            1.0 + self.contrast,
        ).item()

        saturation_factor = torch.empty(1).uniform_(
            max(0.0, 1.0 - self.saturation),
            1.0 + self.saturation,
        ).item()

        hue_factor = torch.empty(1).uniform_(
            -self.hue,
            self.hue,
        ).item()

        # ── Sample ONE horizontal-flip decision for all 9 frames
        apply_flip = (
            torch.rand(1).item() < self.horizontal_flip_p
        )

        transformed_crops = []

        # ── Apply the SAME augmentation parameters to every frame
        for crop in crops:

            # ── Convert PIL Image → Image TVTensor
            crop = self.to_image(crop)

            # ── Apply deterministic spatial preprocessing
            crop = self.resize(crop)
            crop = self.crop(crop)

            # ── Apply the SAME horizontal flip to every frame
            if apply_flip:
                crop = TF.horizontal_flip(crop)

            # ── Apply the SAME rotation to every frame
            crop = TF.rotate(
                crop,
                angle=angle,
            )

            # ── Apply the SAME brightness adjustment to every frame
            crop = TF.adjust_brightness(
                crop,
                brightness_factor,
            )

            # ── Apply the SAME contrast adjustment to every frame
            crop = TF.adjust_contrast(
                crop,
                contrast_factor,
            )

            # ── Apply the SAME saturation adjustment to every frame
            crop = TF.adjust_saturation(
                crop,
                saturation_factor,
            )

            # ── Apply the SAME hue adjustment to every frame
            crop = TF.adjust_hue(
                crop,
                hue_factor,
            )

            # ── Convert uint8 [0, 255] → float32 [0, 1]
            crop = self.to_dtype(crop)

            # ── Normalize using ImageNet mean and standard deviation
            crop = TF.normalize(
                crop,
                mean=self.mean,
                std=self.std,
            )

            transformed_crops.append(crop)

        # ── Stack 9 frames → [9, 3, 224, 224]
        return torch.stack(transformed_crops)


# ── Create training augmentation pipeline
train_transform = TemporalTransform(
    resize_size=256,
    crop_size=224,
    rotation=2.0,
    brightness=0.2,
    contrast=0.2,
    saturation=0.2,
    hue=0.05,
    horizontal_flip_p=0.5,
)

  ```
                               One temporal sample
                                        │
                                        ▼
┌──────────────┬─────────────────┬──────────────┬────────────────┬─────────────┐
│ angle=1.23°  │ brightness=1.08 │ contrast=.95 │ saturation=1.12│ hue=.03     │
│ same × 9     │ same × 9        │ same × 9     │ same × 9       │ same × 9    │ 
└──────────────┴─────────────────┴──────────────┴────────────────┴─────────────┘
                                        │
                                        ▼
                                 F1 F2 F3 F4... F9
                                        │
                                        ▼
                                 [9, 3, 224, 224]
                                        │
                                        ▼
                                       CNN
                                        │
                                        ▼
                                       LSTM
```

In [31]:
class TemporalValTransform:
    """
    Apply deterministic preprocessing to all 9 frames
    of a temporal sequence.

    Output:
        Tensor of shape [9, 3, 224, 224]
    """

    def __init__(
        self,
        resize_size: int = 256,
        crop_size: int = 224,
    ) -> None:

        self.to_image = T.ToImage()

        self.resize = T.Resize(resize_size)
        self.crop = T.CenterCrop(crop_size)

        self.to_dtype = T.ToDtype(
            torch.float32,
            scale=True,
        )

        self.normalize = T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __call__(self, crops: List[Image.Image])-> torch.Tensor:

        transformed_crops = []

        for crop in crops:

            # PIL Image → Image TVTensor
            crop = self.to_image(crop)

            # Deterministic spatial preprocessing
            crop = self.resize(crop)
            crop = self.crop(crop)

            # uint8 → float32 [0, 1]
            crop = self.to_dtype(crop)

            # ImageNet normalization
            crop = self.normalize(crop)

            transformed_crops.append(crop)

        return torch.stack(transformed_crops)


val_transform = TemporalValTransform()

In [32]:
# Number of worker processes used for data loading
# > 0 enables parallel data loading
num_workers = min(4, os.cpu_count() or 1)

# Pin memory is only useful when training on CUDA
# It speeds up CPU → GPU memory transfers
pin_memory = model_config.device == "cuda"

loader_kwargs = dict(
  num_workers = num_workers,
  pin_memory = pin_memory,
  persistent_workers = num_workers > 0,
  # persistent_workers Keeps worker processes alive between epochs
  # Must be False when num_workers == 0
)

# prefetch_factor controls how many batches each worker preloads
# It is only valid when num_workers > 0
if num_workers > 0:
    loader_kwargs["prefetch_factor"] = 2

# Training dataset (uses training split indices, data augmentation, shuffling)
players_train_data = collect_player_temporal_sequences(dataset_root, model_config.train_split)
train_dataset = PlayerTemporalSequenceDataset(players_train_data, train_transform)
train_dataloader = DataLoader(dataset= train_dataset,
                              batch_size= train_config.train_bsize,
                              shuffle= True,
                              **loader_kwargs)

# Validation dataset (no data augmentation, no shuffling)
players_val_data = collect_player_temporal_sequences(dataset_root, model_config.val_split)
val_dataset = PlayerTemporalSequenceDataset(players_val_data, val_transform)
val_dataloader = DataLoader(dataset= val_dataset,
                            batch_size= train_config.val_bsize,
                            shuffle= False,
                            **loader_kwargs)

# Test dataset (used only for final evaluation, no data augmentation, no shuffling)
players_test_data = collect_player_temporal_sequences(dataset_root, model_config.test_split)
test_dataset = PlayerTemporalSequenceDataset(players_test_data, val_transform)
test_dataloader = DataLoader(dataset= test_dataset,
                             batch_size= train_config.val_bsize,
                             shuffle= False,
                             **loader_kwargs)

In [33]:
print(type(train_dataset.transform))
print(type(val_dataset.transform))
print(type(test_dataset.transform))

<class '__main__.TemporalTransform'>
<class '__main__.TemporalValTransform'>
<class '__main__.TemporalValTransform'>


In [34]:
crops, labels = next(iter(train_dataloader))
crops.shape, labels.shape

(torch.Size([32, 9, 3, 224, 224]), torch.Size([32]))

In [35]:
print(f"Number of batches: {len(train_dataloader)}")
train_dataset.summary()

Number of batches: 804
Total Crops       : 231345
Total Samples     : 25705
Unique actions    : 9
----------------------------------------
Player actions frequency:
  standing   : 17500
  moving     : 2376
  waiting    : 1534
  blocking   : 1342
  digging    : 1074
  setting    : 597
  spiking    : 576
  falling    : 539
  jumping    : 167


In [36]:
print(f"Number of batches: {len(val_dataloader)}")
val_dataset.summary()

Number of batches: 500
Total Crops       : 143856
Total Samples     : 15984
Unique actions    : 9
----------------------------------------
Player actions frequency:
  standing   : 10837
  moving     : 1442
  waiting    : 1074
  blocking   : 821
  digging    : 644
  spiking    : 379
  setting    : 356
  falling    : 327
  jumping    : 104


In [37]:
print(f"Number of batches: {len(test_dataloader)}")
test_dataset.summary()

Number of batches: 498
Total Crops       : 143406
Total Samples     : 15934
Unique actions    : 9
----------------------------------------
Player actions frequency:
  standing   : 10819
  moving     : 1352
  waiting    : 1040
  blocking   : 855
  digging    : 659
  falling    : 377
  spiking    : 372
  setting    : 372
  jumping    : 88


# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 6: Model Architecture Design.</div>

In [38]:
class PlayerTemporalClassifier(nn.Module):
  def __init__(self, backbone: nn.Module, num_features:int, num_classes:int, prop_dropout:float) -> None:
    super().__init__()
    # Extract spatial feature maps for each frame.
    # Input : [B*T, C, H, W]
    # Output: [B*T, C, 7, 7]
    self.backbone = backbone

    # Reduce each spatial feature map from 7×7 to 1×1.
    self.adaptive_pool = nn.AdaptiveAvgPool2d((1,1))
    
    # Reduce the CNN feature dimension.
    # Input:  [B*T, num_features]
    # Output: [B*T, 512]
    self.projection = nn.Linear(num_features, 512)

    # Bidirectional LSTM for temporal modeling.
    #
    # Input:  [B, T, 512]
    # Output: [B, T, 512]
    #
    # hidden_size = 256 per direction
    # 256 forward + 256 backward = 512
    self.lstm = nn.LSTM(512, 256, batch_first=True, bidirectional=True)
    
    # Classifier.
    # Input:  [B, 512]
    # Output: [B, num_classes]
    self.classifier = nn.Sequential(
      nn.Linear(512, 256),
      nn.ReLU(),
      nn.Dropout(prop_dropout),
      nn.Linear(256, num_classes)
    )

  def forward(self, x:torch.Tensor)-> torch.Tensor:
    B, T, C, H, W = x.shape

    # Merge batch and temporal dimensions.
    # [B, T, C, H, W] -> [B*T, C, H, W]
    x = x.flatten(0, 1)

    # Extract spatial features using the CNN.
    # [B*T, C, H, W] -> [B*T, num_features, 7, 7]
    x = self.backbone(x)

    # Global average pooling.
    # [B*T, num_features, 7, 7] -> [B*T, num_features, 1, 1]
    x = self.adaptive_pool(x)

    # Remove spatial dimensions.
    # [B*T, num_features, 1, 1] -> [B*T, num_features]
    x = x.flatten(start_dim=1)

    # Project CNN features to a smaller dimension.
    # [B*T, num_features] -> [B*T, 512]
    x = self.projection(x)

    # Restore batch and temporal dimensions.
    # [B*T, 512] -> [B, T, 512]
    x = x.unflatten(0, (B, T))

    # Temporal modeling with a bidirectional LSTM.
    # [B, T, 512] -> [B, T, 512]
    #
    # 256 features from the forward LSTM
    # + 256 features from the backward LSTM
    # = 512 features
    out, (h_n, _) = self.lstm(x)

    
    # Use the output at the last time step.
    # [B, T, 512] -> [B, 512]
    # x = out[:, -1, :]
 
    # or h_n shape: [2, B, 256]
    forward_last = h_n[0]
    backward_last = h_n[1]

    # [B, 256] + [B, 256] -> [B, 512]
    x = torch.cat((forward_last, backward_last), dim=1)

    # Classification.
    # [B, 512] -> [B, num_classes]
    logits = self.classifier(x)

    return logits

In [39]:
# ── Initialize the model,loss function, optimizer, and learning rate scheduler.

weights = models.ResNet50_Weights.DEFAULT
original_model = models.resnet50(weights=weights)

layers = list(original_model.children())[:-2]
truncated_model = nn.Sequential(*layers)

# Determine the backbone feature dimension
with torch.no_grad():
  dummy_data = torch.zeros((1,3,224,224))
  num_features = truncated_model(dummy_data).shape[1]

# ── Model ─────────────────────────────────────────────────────────────────────
BaseLine5_a_model = PlayerTemporalClassifier(
  truncated_model,
  num_features,
  model_config.player_classes,
  model_config.dropout_prop
)

# ── Loss Function ─────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(reduction="none", label_smoothing=0.15)

# ── Optimizer ─────────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
  params = BaseLine5_a_model.parameters(),
  lr = train_config.learning_rate,
  betas = train_config.betas,
  weight_decay = train_config.weight_decay
)

# ── Learning Rate Scheduler ───────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
  optimizer = optimizer,
  T_max = train_config.epochs,
  eta_min = train_config.eta_min
)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 110MB/s] 


In [40]:
try:
  import torchinfo
  print(f"trochinfo version: {torchinfo.__version__}")
except:
  !pip install torchinfo >/dev/null 2>&1
  import torchinfo
  print(f"torchinfo version: {torchinfo.__version__}")

dummy_test_data = torch.zeros((1, 9, 3, 224, 224))
torchinfo.summary(model= BaseLine5_a_model, input_data=dummy_test_data, device = "cpu")

trochinfo version: 1.8.0


Layer (type:depth-idx)                        Output Shape              Param #
PlayerTemporalClassifier                      [1, 9]                    --
├─Sequential: 1-1                             [9, 2048, 7, 7]           --
│    └─Conv2d: 2-1                            [9, 64, 112, 112]         9,408
│    └─BatchNorm2d: 2-2                       [9, 64, 112, 112]         128
│    └─ReLU: 2-3                              [9, 64, 112, 112]         --
│    └─MaxPool2d: 2-4                         [9, 64, 56, 56]           --
│    └─Sequential: 2-5                        [9, 256, 56, 56]          --
│    │    └─Bottleneck: 3-1                   [9, 256, 56, 56]          75,008
│    │    └─Bottleneck: 3-2                   [9, 256, 56, 56]          70,400
│    │    └─Bottleneck: 3-3                   [9, 256, 56, 56]          70,400
│    └─Sequential: 2-6                        [9, 512, 28, 28]          --
│    │    └─Bottleneck: 3-4                   [9, 512, 28, 28]          379,392

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 7: Training and Testing Pipeline.</div>

In [41]:
import random
import numpy as np

def set_all_seeds(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)

In [42]:
def accuracy_fn(logits: torch.Tensor, targets: torch.Tensor)->Tuple[int, int]:
  """
    Calculates the number of correct predictions and batch size.

    Args:
        logits (torch.Tensor): Raw model outputs, shape (B, num_classes)
        targets (torch.Tensor): Ground truth labels, shape (B,)

    Returns:
        Tuple[int, int]:
            - batch_correct: number of correct predictions in this batch
            - batch_size: number of samples in this batch
  """
  preds = logits.argmax(dim=1)  # Go from logits -> pred labels
  batch_correct = (preds == targets).sum().item()
  batch_size = targets.size(0)
  return batch_correct, batch_size

In [43]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               accuracy_fn: Callable[[torch.Tensor, torch.Tensor], Tuple[int, int]],
               criterion : torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               scaler: torch.cuda.amp.GradScaler,
               device: torch.device
              )-> dict[str, Any]:
  """
    Perform one training epoch.

    Notes:
        - `criterion` must return the per-sample loss (reduction="none").
        - Input shape is expected to be (B, T, C, H, W).
  """
  total_train_loss, total_correct = 0.0, 0
  total_samples = 0

  model.train()
  for crops, targets in dataloader:
    crops = crops.to(device, non_blocking = True)
    targets = targets.to(device, non_blocking =True)

    optimizer.zero_grad(set_to_none=True) # faster & safer

    # Forward Pass
    with torch.autocast(
      device_type=device.type,
      enabled= device.type == "cuda"
    ):

      logits = model(crops)

      per_sample_loss = criterion(logits, targets)
      loss = per_sample_loss.mean()

    # Backward Pass
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    # Accumulate loss
    total_train_loss += loss.item() * targets.size(0)

    # Calculate batch accuracy
    batch_correct, batch_size = accuracy_fn(logits, targets)

    total_correct += batch_correct
    total_samples += batch_size

  # Epoch-level metrics
  avg_loss = total_train_loss / total_samples
  avg_acc = (total_correct / total_samples) * 100

  return{
    "Model_Name": model.__class__.__name__,
    "Train_loss": avg_loss,
    "Train_acc": avg_acc
  }

In [44]:
def eval_step(model:torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              accuracy_fn: Callable[[torch.Tensor, torch.Tensor], Tuple[int, int]],
              criterion: torch.nn.Module,
              device: torch.device
             )-> dict[str, Any]:
  """
    Evaluate the model on the validation/test dataset and compute loss, accuracy
    and Macro F1 score.

    Returns:
        dict[str, Any]: Evaluation metrics including model name, average loss
        and accuracy (%), and Macro F1 score.
  """
  total_loss, total_correct = 0.0, 0
  all_targets, all_preds = [], []
  total_samples = 0

  model.eval()

  # Disable gradient computation during evaluation
  with torch.inference_mode():
    for crops, targets in dataloader:
      crops = crops.to(device, non_blocking = True)
      targets = targets.to(device, non_blocking =True)

      # Forward Pass
      with torch.autocast(
        device_type=device.type,
        enabled= device.type == "cuda"):

        logits = model(crops)
        per_sample_loss = criterion(logits, targets)
        loss = per_sample_loss.mean()

      # Accumulate loss
      total_loss += loss.item() * targets.size(0)

      # Calculate batch accuracy
      batch_correct, batch_size = accuracy_fn(logits, targets)
      total_correct += batch_correct
      total_samples += batch_size

      # Convert logits to predicted class labels
      preds = logits.argmax(dim=1)


      # Store predictions and targets for epoch-level Macro F1
      # Move to CPU because scikit-learn expects CPU/NumPy arrays
      all_preds.append(preds.cpu())
      all_targets.append(targets.cpu())

    # Epoch-level metrics
    avg_loss = total_loss / total_samples
    avg_acc = (total_correct / total_samples) * 100

    # Combine predictions and targets from all batches
    all_targets = torch.cat(all_targets)
    all_preds = torch.cat(all_preds)

    # Compute Macro F1 over the entire dataset
    macro_f1 = f1_score(
      y_true= all_targets.numpy(),
      y_pred= all_preds.numpy(),
      average="macro"
    )

  return{
    "Model_Name": model.__class__.__name__,
    "macro_f1": macro_f1,
    "loss": avg_loss,
    "acc": avg_acc,
  }

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 8: Training and Validation Model With Mixed Precision.</div>

In [ ]:
"""
Run the complete training pipeline, including training, validation,
checkpointing, scheduler updates, and metric logging.
"""

from torch.cuda.amp import  GradScaler # PERFORMANCE OPTIMIZATIONS
scaler = GradScaler()  # helps scale gradients safely

set_all_seeds(42)
train_history, val_history = {}, {}
train_acc, val_acc, macro_f1 = [], [], []
epochs, val_losses, train_losses, lr_history = [], [], [], []

best_epoch=0
best_macro_f1 = 0.0

model = BaseLine5_a_model
model.to(model_config.device)

for epoch in range(1, train_config.epochs + 1):
  # Train → Validate → Compute Macro F1 → Save if best → Scheduler step

  # ── Train ───────────────────────────────────────────────────────────────────
  train_history = train_step(model= model,
                             dataloader= train_dataloader,
                             accuracy_fn= accuracy_fn,
                             criterion= criterion,
                             optimizer= optimizer,
                             scaler= scaler,
                             device= model_config.device)

  # ── Validate ────────────────────────────────────────────────────────────────
  val_history = eval_step(model= model,
                          dataloader = val_dataloader,
                          accuracy_fn= accuracy_fn,
                          criterion= criterion,
                          device= model_config.device)

  # ── Logging ─────────────────────────────────────────────────────────────────
  train_losses.append(train_history['Train_loss'])
  train_acc.append(train_history['Train_acc'])

  val_losses.append(val_history['loss'])
  val_acc.append(val_history['acc'])

  lr_history.append(optimizer.param_groups[0]['lr'])
  macro_f1.append(val_history['macro_f1'])
  epochs.append(epoch)

  # ── Checkpoint ──────────────────────────────────────────────────────────────
  if val_history["macro_f1"] > best_macro_f1:
    best_epoch = epoch
    best_macro_f1 = val_history["macro_f1"]
    save_checkpoints(model,
                     scaler,
                     optimizer,
                     train_history["Train_loss"],
                     val_history["loss"],
                     best_macro_f1,
                     epoch,
                     os.path.join(train_config.checkpoint_path, f"checkpoint_epoch_{best_epoch}.pt"))

  # ── SCHEDULER ───────────────────────────────────────────────────────────────
  # CosineAnnealingLR check per epoch or batch, always after validation.
  # Which depends on the scheduler type.
  scheduler.step()

  # ── PRINT ───────────────────────────────────────────────────────────────────
  if(epoch) % train_config.print_every == 0:
    print(f"Epoch [{epoch:>2}] - Macro F1: {val_history['macro_f1']:.3f} | Train loss: {train_history['Train_loss']:.3f} | Train acc: {train_history['Train_acc']:.3f}% | Eval loss: {val_history['loss']:.3f} | Eval acc: {val_history['acc']:.3f}% | LR: {scheduler.get_last_lr()[0]:.6f}")

Saved checkpoint: /kaggle/working/chekpoints/checkpoint_epoch_1.pt
Epoch [ 1] - Macro F1: 0.399 | Train loss: 1.316 | Train acc: 71.216% | Eval loss: 1.215 | Eval acc: 74.337% | LR: 0.000293
Saved checkpoint: /kaggle/working/chekpoints/checkpoint_epoch_2.pt
Epoch [ 2] - Macro F1: 0.431 | Train loss: 1.185 | Train acc: 76.051% | Eval loss: 1.187 | Eval acc: 75.638% | LR: 0.000271
Saved checkpoint: /kaggle/working/chekpoints/checkpoint_epoch_3.pt
Epoch [ 3] - Macro F1: 0.477 | Train loss: 1.137 | Train acc: 78.203% | Eval loss: 1.156 | Eval acc: 76.458% | LR: 0.000238
Saved checkpoint: /kaggle/working/chekpoints/checkpoint_epoch_4.pt
Epoch [ 4] - Macro F1: 0.520 | Train loss: 1.096 | Train acc: 79.883% | Eval loss: 1.121 | Eval acc: 78.172% | LR: 0.000197
Epoch [ 5] - Macro F1: 0.509 | Train loss: 1.058 | Train acc: 81.587% | Eval loss: 1.116 | Eval acc: 78.034% | LR: 0.000150
Saved checkpoint: /kaggle/working/chekpoints/checkpoint_epoch_6.pt
Epoch [ 6] - Macro F1: 0.531 | Train loss: 1.

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Step 9: Evaluate Model.</div>

In [ ]:
model = PlayerTemporalClassifier(
    backbone=truncated_model,
    num_features= num_features,
    num_classes = model_config.player_classes,
    prop_dropout= model_config.dropout_prop 
)

ckpt = load_checkpoint(path = f"/kaggle/working/chekpoints/checkpoint_epoch_{best_epoch}.pt", device=model_config.device)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(model_config.device)

In [ ]:
Model_result = eval_step(model, test_dataloader, accuracy_fn, criterion, model_config.device)
Model_result

In [ ]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              device: torch.device
              )-> Tuple[np.ndarray, np.ndarray]:
  """
    Evaluate the model on a dataset and return ground-truth and predicted labels.

    Performs inference without gradient tracking, collects predictions for all
    batches, and returns the concatenated ground-truth and predicted labels as
    NumPy arrays for downstream evaluation (e.g. classification_report).

    Args:
        model (torch.nn.Module): Trained model to evaluate.
        dataloader (torch.utils.data.DataLoader): DataLoader providing
            evaluation batches.
        device (torch.device): Device used for inference.

    Returns:
        Tuple[np.ndarray, np.ndarray]:
            - y_true: Ground-truth labels.
            - y_pred: Predicted class labels.
  """

  y_true, y_pred = [], []

  # ── Evaluation Mode ─────────────────────────────────────────────────────────
  model.eval()
  with torch.inference_mode():
    for crops, targets in dataloader:

      # ── Move Batch to Device ────────────────────────────────────────────────
      crops = crops.to(device, non_blocking=True)
      targets = targets.to(device, non_blocking=True)

      # ── Forward Pass ────────────────────────────────────────────────────────
      with torch.autocast(
        device_type=device.type, enabled=device.type == "cuda"):

        logits = model(crops)

       # ── Collect Predictions ────────────────────────────────────────────────
      preds = logits.argmax(dim=1)

      y_pred.append(preds.cpu())
      y_true.append(targets.cpu())

    # ── Concatenate Results ───────────────────────────────────────────────────
    y_pred_np = torch.cat(y_pred).numpy()
    y_true_np = torch.cat(y_true).numpy()

  return y_true_np, y_pred_np

# ── Test Model ────────────────────────────────────────────────────────────────
y_true, y_pred = test_step(model= model,
                            dataloader= test_dataloader,
                            device= model_config.device)

In [ ]:
def display_train_eval_curves(epochs:List[int],
                              train_losses:List[float],
                              val_losses:List[float],
                              train_acc:List[float],
                              val_acc:list[float])->None:
  """
    Plot training and validation loss and accuracy curves over epochs.

    Returns:
        None: Displays the training and validation curves.
  """

  plt.figure(figsize = (12,5), dpi = 100)

  # ── Loss Curves ─────────────────────────────────────────────────────────────
  plt.subplot(1,2,1)
  plt.plot(epochs, train_losses, label = "Train Loss")
  plt.plot(epochs, val_losses, label = "val Loss")
  plt.ylabel("Loss", fontsize= 12, weight='bold')
  plt.xlabel("Epochs", fontsize= 12, weight='bold')
  plt.title("Training and Validation Loss", fontsize=12, weight="bold")
  plt.legend(prop = {"size":12},loc = "best")
  plt.grid(True)


  # ── Accuracy Curves ─────────────────────────────────────────────────────────
  plt.subplot(1,2,2)
  plt.plot(epochs,train_acc,label = "Train acc")
  plt.plot(epochs ,val_acc, label = "val acc")
  plt.ylabel("Accuracy" ,fontsize = 12 ,weight='bold')
  plt.xlabel("Epochs" ,fontsize = 12 ,weight='bold')
  plt.title("Training and Validation Accuracy", fontsize=12, weight="bold")
  plt.legend(prop = {"size":12},loc = "best")
  plt.grid(True)

  plt.tight_layout()
  # plt.axis("off")
  plt.show()


def display_classification_report(y_true:np.ndarray,
                                  y_pred:np.ndarray,
                                  classes_names:List[str])->None:
  """
    Display the classification report for model predictions.

    Args:
        y_true (np.ndarray): Ground-truth labels.
        y_pred (np.ndarray): Predicted labels.
        class_names (List[str]): Class names corresponding to label indices.

    Returns:
        None: Prints the classification report.
  """

  # ── Classification Report ───────────────────────────────────────────────────
  report = classification_report(
    y_true = y_true,
    y_pred= y_pred,
    target_names= classes_names
  )

  print(f"Classification Report:\n{report}")

In [ ]:
def display_macrof1_curve(epoch_values: List[int],
                          macro_f1_values: List[float],
                          save_path: Optional[str] = None,
                          show: bool = True
                          )->None:
  """
    Plot the Macro F1 score over training epochs.

    Returns:
        None: Displays and/or saves the Macro F1 curve.
  """

  # ── Create Figure ───────────────────────────────────────────────────────────
  fig = plt.figure(figsize=(10, 5), dpi=100)

  # ── Macro F1 Curve ──────────────────────────────────────────────────────────
  plt.subplot(1,2,1)
  plt.plot(
    epoch_values,
    macro_f1_values,
    color='orange',
    marker="o",
    linewidth=2,
    label="Macro F1"
  )

  plt.title("Macro F1 over Epochs", fontsize=12, weight="bold")
  plt.xlabel("Epochs", fontsize=12, weight="bold")
  plt.ylabel("Macro F1", fontsize=12, weight="bold")
  plt.legend(prop={"size": 12}, loc="best")
  plt.grid(True)

  plt.tight_layout()

  # ── Save Figure ─────────────────────────────────────────────────────────────

  if save_path is not None:
      os.makedirs(os.path.dirname(save_path), exist_ok=True)
      plt.savefig(save_path, dpi=300, bbox_inches="tight")

  # ── Display Figure ──────────────────────────────────────────────────────────

  if show:
    plt.show()

  else:
    plt.close(fig)

In [ ]:
try:
  import torchmetrics, mlxtend
  print(f"mlxtend version: {mlxtend.__version__}")
  assert int(mlxtend.__version__.split(".")[1])>=19, "mlxtend verison should be 0.19.0 or higher"

except Exception as e:
  ! pip install torchmetrics > /dev/null 2>&1 && pip install -U mlxtend >/dev/null 2>&1
  import torchmetrics, mlxtend

  print(f"mlxtend version: {mlxtend.__version__}")

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix
def display_confusion_matrix(y_true: np.ndarray,
                             y_preds:np.ndarray,
                             classes_names:List[str])-> None:
  """
    Display the confusion matrix for multiclass classification.

    Args:
        y_true (np.ndarray): Ground-truth labels.
        y_pred (np.ndarray): Predicted labels.
        class_names (List[str]): Class names corresponding to label indices.

    Returns:
        None: Displays the confusion matrix.
  """

  # ── Convert to PyTorch Tensors ──────────────────────────────────────────────
  # Convert NumPy arrays to PyTorch tensors (shares memory, avoids copying data)
  y_true = torch.from_numpy(y_true)
  y_preds = torch.from_numpy(y_preds)

  # ── Compute Confusion Matrix ────────────────────────────────────────────────
  confmat = ConfusionMatrix(
    task='multiclass',
    num_classes=len(classes_names),
  )

  confmat_tensor = confmat(
    preds= y_preds,
    target= y_true
  ).cpu().numpy()


  # ── Plot Confusion Matrix ──────────────────────────────────────────────────
  fig, ax = plot_confusion_matrix(conf_mat = confmat_tensor,
                                  class_names= classes_names,
                                  figsize = (10,5))
  plt.tight_layout()
  plt.show()

In [ ]:
display_classification_report(y_true, y_pred, model_config.classes_names)

In [ ]:
display_confusion_matrix(y_true, y_pred, model_config.classes_names)

In [ ]:
display_macrof1_curve(epochs,
                      macro_f1,
                      r"/kaggle/working/Figures")

In [ ]:
display_train_eval_curves(epochs,
                          train_losses, val_losses, train_acc, val_acc)

# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Thanks & Upvote ❤️</div>